# Can we predict 12-month loan default?
Follows `checklist.md`. Target: `defaulted`.

**Data facts that drive the design**
- `application_id` is unique per row (no entity repeats) and is an identifier -> dropped. Ids are not monotone in row order and there is no date column, so there is no time ordering to respect. A stratified random split is therefore the appropriate structure (whole-entity grouping is automatically satisfied since each entity is one row).
- All features are recorded at application, i.e. before funding, so all are known at prediction time. `defaulted` is the outcome.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
df=pd.read_csv('data.csv')
print(df.shape, 'dup ids:',df.application_id.duplicated().sum(), 'missing:',df.isna().sum().sum())
print('default rate:',df.defaulted.mean())

## 1. Set aside the test set (before any modelling or looking at relationships)

In [ ]:
train,test=train_test_split(df,test_size=0.2,stratify=df.defaulted,random_state=0)
FEATS=['region','income_k','debt_ratio','credit_lines','age','employment_years','loan_amount_k']
NUM=[f for f in FEATS if f!='region']
Xtr,ytr=train[FEATS],train.defaulted
Xte,yte=test[FEATS],test.defaulted
print(len(train),len(test),ytr.mean(),yte.mean())

## 2-4. Candidates (named before scoring) and CV structure
Candidates, all declared up front:
1. `baseline`: predicts the training prevalence (trivial).
2. `logreg_C0.1`, `logreg_C1`, `logreg_C10`: standardised numerics + one-hot region.
3. `hgb_small`: gradient boosting, depth 3, 100 iters, lr 0.05.
4. `hgb_large`: gradient boosting, depth 6, 300 iters, lr 0.05.

Metric: ROC AUC (primary), average precision (secondary, since the 10% positive rate is imbalanced). Scaling/encoding live inside the Pipeline, so they are fit within each CV fold. CV is 5-fold stratified on training rows only (one row per entity, so folds are entity-disjoint).

In [ ]:
pre=lambda: ColumnTransformer([('n',StandardScaler(),NUM),('c',OneHotEncoder(handle_unknown='ignore'),['region'])])
def lr(C): return Pipeline([('p',pre()),('m',LogisticRegression(C=C,max_iter=1000))])
def hgb(**k): return Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(learning_rate=0.05,random_state=0,**k))])
candidates={
 'baseline':Pipeline([('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':lr(0.1),'logreg_C1':lr(1),'logreg_C10':lr(10),
 'hgb_small':hgb(max_depth=3,max_iter=100),
 'hgb_large':hgb(max_depth=6,max_iter=300),
}
cv=StratifiedKFold(5,shuffle=True,random_state=1)
oof={};rows=[]
for n,m in candidates.items():
    p=cross_val_predict(m,Xtr,ytr,cv=cv,method='predict_proba')[:,1]; oof[n]=p
    rows.append((n,roc_auc_score(ytr,p),average_precision_score(ytr,p)))
cvres=pd.DataFrame(rows,columns=['model','cv_auc','cv_ap']).sort_values('cv_auc',ascending=False);cvres

## 5. Choose model and decision threshold on CV only

In [ ]:
best=cvres.iloc[0].model
p=oof[best]
ths=np.linspace(0.02,0.6,59)
f1s=[f1_score(ytr,p>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| CV F1-optimal threshold:',round(thr,3),'| CV F1:',round(max(f1s),3))

**Decision recorded before touching the test set:** the model with the highest CV AUC above (`best`), with the CV F1-optimal threshold `thr`. Nothing about the test set informs this choice.

## 6. Refit on all training rows, score the test set once

In [ ]:
final=candidates[best].fit(Xtr,ytr)
pt=final.predict_proba(Xte)[:,1]
auc=roc_auc_score(yte,pt)
rng=np.random.default_rng(0);bs=[]
for _ in range(1000):
    i=rng.integers(0,len(yte),len(yte))
    if yte.iloc[i].nunique()==2: bs.append(roc_auc_score(yte.iloc[i],pt[i]))
print(f'HEADLINE test AUC = {auc:.3f}  (95% bootstrap CI {np.percentile(bs,2.5):.3f}-{np.percentile(bs,97.5):.3f})')
print(f'test average precision = {average_precision_score(yte,pt):.3f} (prevalence {yte.mean():.3f})')
print(f'test F1 at CV-chosen threshold {thr:.2f} = {f1_score(yte,pt>=thr):.3f}')

## Interpretation
See the printed headline above. The baseline's CV AUC is ~0.5, so any lift over it reflects real signal. The test score is reported once and was not used for any selection. Caveats: no date column, so we could not test robustness to time shift; if the model is used on future applications, performance could differ.